# GLCM Texture Metrics

Gray-Level Co-occurrence Matrix (GLCM) texture features capture spatial patterns in raster data that spectral bands alone cannot distinguish. They were introduced by Haralick et al. (1973) and remain a standard tool in remote sensing classification, geological mapping, and medical imaging.

`xrspatial.glcm_texture` computes six Haralick features over a sliding window:

| Metric | What it measures |
|---|---|
| **contrast** | Intensity difference between neighboring pixels |
| **dissimilarity** | Mean absolute gray-level difference |
| **homogeneity** | Inverse difference moment (smooth vs. rough) |
| **energy** | Sum of squared GLCM entries (uniformity) |
| **correlation** | Linear dependency of gray levels |
| **entropy** | Randomness of the co-occurrence distribution |

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from xrspatial import glcm_texture

## Synthetic test raster

We'll build a 100x100 raster with four quadrants that have distinct textures: smooth, noisy, striped, and checkerboard. GLCM metrics should clearly separate these regions.

In [ ]:
# Build a 100x100 raster with four texture quadrants
rng = np.random.default_rng(42)
data = np.zeros((100, 100), dtype=np.float64)

# Top-left: smooth gradient
data[:50, :50] = np.linspace(0, 1, 50)[np.newaxis, :]

# Top-right: random noise
data[:50, 50:] = rng.random((50, 50))

# Bottom-left: vertical stripes
data[50:, :50] = np.tile([0.0, 1.0], 25)[np.newaxis, :]

# Bottom-right: checkerboard
cb = np.zeros((50, 50))
cb[::2, 1::2] = 1.0
cb[1::2, ::2] = 1.0
data[50:, 50:] = cb

agg = xr.DataArray(data, dims=['y', 'x'])

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(data, cmap='gray')
ax.set_title('Input raster (4 texture regions)')
plt.tight_layout()
plt.show()

## Computing a single metric

The simplest usage: pass a single metric name and get a 2-D result back.

In [ ]:
contrast = glcm_texture(agg, metric='contrast', window_size=7, levels=64)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(data, cmap='gray')
axes[0].set_title('Input')
im = axes[1].imshow(contrast.values, cmap='inferno')
axes[1].set_title('GLCM Contrast')
plt.colorbar(im, ax=axes[1], shrink=0.8)
plt.tight_layout()
plt.show()

## Computing multiple metrics at once

Pass a list of metric names to get a 3-D result with a leading `metric` dimension. This is more efficient than calling `glcm_texture` separately for each metric when using the numpy backend, since the GLCM is built once per window position.

In [ ]:
metrics = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'entropy']
textures = glcm_texture(agg, metric=metrics, window_size=7, levels=64)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, name in zip(axes.flat, metrics):
    vals = textures.sel(metric=name).values
    im = ax.imshow(vals, cmap='viridis')
    ax.set_title(name.capitalize())
    plt.colorbar(im, ax=ax, shrink=0.7)
plt.suptitle('All six GLCM texture metrics', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Effect of angle

The `angle` parameter controls the direction of pixel pairing:
- `0` -- horizontal (right)
- `45` -- upper-right diagonal
- `90` -- vertical (up)
- `135` -- upper-left diagonal
- `None` (default) -- average over all four

Directional textures like stripes respond differently depending on the angle.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, ang in zip(axes, [0, 45, 90, 135]):
    result = glcm_texture(agg, metric='contrast', window_size=7,
                          levels=64, angle=ang)
    im = ax.imshow(result.values, cmap='inferno')
    ax.set_title(f'Contrast (angle={ang})')
    plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

## Dask support

`glcm_texture` works on chunked Dask arrays out of the box. The input is quantized globally (so gray-level mapping stays consistent across chunks), then each chunk computes GLCM features independently via `map_overlap`.

In [ ]:
import dask.array as da

dask_agg = xr.DataArray(
    da.from_array(data, chunks=(50, 50)),
    dims=['y', 'x'],
)

dask_contrast = glcm_texture(dask_agg, metric='contrast',
                             window_size=7, levels=64)
print(f'Result type: {type(dask_contrast.data)}')
print(f'Chunks: {dask_contrast.data.chunks}')

# Verify it matches the numpy result
np.testing.assert_allclose(
    contrast.values, dask_contrast.values,
    rtol=1e-10, equal_nan=True,
)
print('Dask result matches numpy result.')

## Parameters reference

| Parameter | Default | Description |
|---|---|---|
| `metric` | `'contrast'` | One metric name (str) or a list of names |
| `window_size` | `7` | Side length of the sliding window (must be odd, >= 3) |
| `levels` | `64` | Number of gray levels for quantization (2-256) |
| `distance` | `1` | Pixel pair distance |
| `angle` | `None` | 0, 45, 90, 135, or None (average all four) |

Lower `levels` values run faster but lose gray-level resolution. For most remote sensing work, 32-64 levels is a good balance.